In [2]:
# Import packages 
import pandas as pd
import sys
from collections import defaultdict, OrderedDict
import glob
import os
import shutil
import numpy as np
# import seaborn as sns
# import matplotlib.colors as mcolors
# from matplotlib.colors import LinearSegmentedColormap
import glob
import sys
import re
print(sys.version)
sys.path.append('/Users/geba9152/LIET/liet/')

from liet_res_class import FitParse, fitparse_intersect
import rnap_lib_data_sim as ds
import rnap_lib_plotting as rp
from rnap_lib_fitting_results import results_loader, pdf_generator, hist_generator
from functools import reduce


# LIET library packages
sys.path.append('/Users/geba9152/LIET/liet/')
import rnap_lib_data_proc as dp
from liet_res_class import FitParse, fitparse_intersect
from rnap_lib_fitting_results import results_loader
import rnap_lib_plotting as rp

sys.path.append('/Users/geba9152/fall_2024_3prime/')
from load_in_liet import pull_out_params_meta_samples

3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]


# Find Orthologs Across Species To Run LIET On

* 1st, run BUSCO (see `busco.sbatch`)
* 2nd, look for overlaps (see script below)
* 3rd, filter overlaps (see code block below)

```
#!/bin/bash

#SBATCH --nodes=1
#SBATCH --ntasks=2 # Num of CPU                                                                                                  
#SBATCH --time=24:00:00 # Time limit                                                                                                                
#SBATCH --partition short
#SBATCH --mem=4gb # Memory                                                                                                                                 
#SBATCH --output=/scratch/Shares/dowell/for_RU/hg-38-busco-run-01162024/e_and_o/%x_%j.out
#SBATCH --error=/scratch/Shares/dowell/for_RU/hg-38-busco-run-01162024/e_and_o/%x_%j.err                                                                 
#SBATCH --mail-type=FAIL
#SBATCH --mail-user=rutendo.sigauke@colorado.edu

###########################
# Process Metadata        #
###########################
printf "Run on: %s\n" "$(hostname)"
printf "Run from: %s\n" "$(pwd)"
printf "Script: %s\n" "$0"
date

###########################
# Load modules            #
###########################
module load bedtools/2.28.0

###########################
# Define paths            #
###########################
wd=/scratch/Shares/dowell/for_RU/hg-38-busco-run-01162024
hg38_bed=/scratch/Shares/dowell/genomes/hg38/ncbi/hg38_refseq_diff53prime_with_putatives.bed
buscos=${wd}/full_table.tsv

###########################
# Get overlapping regions #
###########################

#1: convert buscos to bed file format
#2: removing any missing fragments, since those won't have any genome coordinates
#-for negative strand transcripts make sure the start is the smallest coordinate
cat ${buscos} | \
    grep -v "#" | \
    awk 'OFS="\t" {if ($2!="Missing"&& $6=="+"){print $3,$4,$5,$1,".",$6}}' > \
    ${wd}/hg38_busco_hits_pos.bed

cat ${buscos} | \
    grep -v "#" | \
    awk 'OFS="\t" {if ($2!="Missing"&& $6=="-"){print $3,$5,$4,$1,".",$6}}' > \
    ${wd}/hg38_busco_hits_neg.bed
  
#3: use bedtools to get overlaps with genes
#4: overlap is done in a strand specific manner
#5: returning all busco and refseq features
cat ${wd}/hg38_busco_hits_pos.bed ${wd}/hg38_busco_hits_neg.bed | \
    sort -k 1,1 -k2,2n | \
    bedtools intersect -a - -b ${hg38_bed} -s -wo > ${wd}/hg38_busco_refseq_diff53prime_with_putatives.bed

#5: remove intermediate files
rm ${wd}/hg38_busco_hits_pos.bed
rm ${wd}/hg38_busco_hits_neg.bed

echo "Done!"

```

In [2]:
# 1. Loop through busco overlap files, filter for genes that overlap by >= 75%, intersect lists across species 
indir='/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/ortholog-output'

def get_gene_orthologs_across_species(indir):
    # Initialize an empty list to store sets of genes from each file
    gene_sets = []

    # Loop through all files in the directory
    for file in os.listdir(indir):

         if file.endswith(".bed"):  # Process only .bed files
            filepath = os.path.join(indir, file)

            # Read the BED file
            df = pd.read_csv(filepath, sep="\t", header=None)

            # Gene name col
            df[13] = df[9].str.split(":").str[0]
            
            # Gene length col BUSCO
            df[14] = abs(df[1] - df[2])

            # Determine if Refseq overlaps (bps) is >= 75% of the gene length col BUSCO (bases)
            # The column below is the proportion overlap (1 = 100%)
            df[15] = df[12]/df[14]

            # Filter for genes that overlap by >=75% gene length
            df = df[df[15] >= 0.75]

            # Select unique genes
            unique_genes = set(df[13].drop_duplicates())
            
            # Append to list of gene sets
            # we now have a list of sets
            gene_sets.append(unique_genes)

    # Find genes that overlap across all files
    # "reduce" finds the common elements (intersection) of multiple sets
    common_genes = reduce(set.intersection, gene_sets)
    print(len(common_genes))
    
    return common_genes

common_genes = get_gene_orthologs_across_species(indir)
common_genes

3436


{'FOXM1',
 'ARID4A',
 'DMRTA1',
 'NR4A2',
 'WDR26',
 'SPRED1',
 'EIF2B3',
 'FAM131B',
 'CCDC81',
 'BNC1',
 'FUT7',
 'MRPL1',
 'ZGRF1',
 'ERRFI1',
 'SLC43A3',
 'RCAN2',
 'EIF2S2',
 'SECISBP2L',
 'USH2A',
 'CHRNG',
 'NAA50',
 'MN1',
 'LY86',
 'PEX11G',
 'RAD17',
 'VPS4B',
 'OXGR1',
 'NUP210L',
 'CYTIP',
 'FOS',
 'GIN1',
 'TXNDC17',
 'NPC2',
 'SFXN2',
 'LRRTM3',
 'RNF31',
 'LPAR2',
 'SLC46A2',
 'TRAPPC1',
 'CTNNBIP1',
 'DGUOK',
 'MC5R',
 'DTHD1',
 'FBXO41',
 'KCNQ5',
 'TMEM38B',
 'ZFYVE1',
 'BMPR1A',
 'ADAMTS5',
 'MLF2',
 'ZMYND12',
 'NR3C1',
 'AP4S1',
 'UAP1',
 'NDRG2',
 'R3HCC1L',
 'NXPH2',
 'PELP1',
 'TBC1D24',
 'RFC1',
 'NENF',
 'FBXO40',
 'PLA2G2E',
 'ZFAND6',
 'TSPAN2',
 'YIPF1',
 'ADCK1',
 'CLIP4',
 'PDCL2',
 'CCDC80',
 'CARTPT',
 'STK24',
 'F2',
 'WLS',
 'RAD51AP1',
 'LARP7',
 'MLPH',
 'B4GALT2',
 'TMF1',
 'TMEM207',
 'PDE2A',
 'NUFIP2',
 'BCAP29',
 'PTRHD1',
 'STMND1',
 'TMEM62',
 'ZBTB5',
 'LRRC40',
 'FLRT2',
 'ZBTB11',
 'MYRFL',
 'LCP1',
 'NFATC4',
 'GALNT16',
 'SLC6A4',
 'JOSD

In [3]:
# 2. Overlap with MANE transcripts/get coorinates to "lift over" using cactus

MANE_transcripts = '/scratch/Shares/dowell/geba9152/MANE/transcript.MANE.refseq.bed'
MANE_transcripts = pd.read_csv(MANE_transcripts, sep = "\t", header = None)
MANE_transcripts[6] = MANE_transcripts[3].str.split("|").str[0]

# Overlapping genes with common genes
overlap_orthologs = MANE_transcripts[MANE_transcripts[6].isin(common_genes)]
print(len(overlap_orthologs))

# Identify and sort duplicates (same gene diff isoform)
duplicates = overlap_orthologs[overlap_orthologs.duplicated(subset=6, keep=False)]
duplicates = duplicates.sort_values(by=6)
print(len(duplicates))

# List of duplicate genes to remove from overlap_orthologs df
genes_to_remove = duplicates[3].to_list()

# Remove from overlap_orthologs for now (will select an isoform if start/stop coorinates are same, eg: only diff is in introns/exons)
overlap_orthologs = overlap_orthologs[~overlap_orthologs[3].isin(genes_to_remove)]
print(len(overlap_orthologs))

# Now let's get the filtered duplicates we can add back
filtered_duplicates = duplicates[
    duplicates.duplicated(subset=[1, 2], keep=False)
]

# Drop duplicates (doesnt matter which isoform because start/stop are the same)
filtered_duplicates = filtered_duplicates[~filtered_duplicates.duplicated(subset=6, keep='first')]
print(len(filtered_duplicates))

# Add back into the overlap orthologs
overlap_orthologs = pd.concat([overlap_orthologs, filtered_duplicates])
overlap_orthologs = overlap_orthologs[[0,1,2,3,4,5]]
# overlap_orthologs[3] = overlap_orthologs[3].str.replace("|",":")
overlap_orthologs = overlap_orthologs.sort_values(by=[0, 1])


overlap_orthologs[3] = overlap_orthologs[3].str.split("|").str[0]
overlap_orthologs = overlap_orthologs.sort_values(by=[0, 1])
overlap_orthologs[6] = 0
overlap_orthologs = overlap_orthologs[[0,1,2,3,6,5]]
# Save to bed file for cactus
outdir='/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/ortholog-output/MANE-annotation-overlaps-across-all-species/overlapping_orthologs_Hal_format.MANE.bed'
overlap_orthologs.to_csv(outdir, sep = "\t", header = None, index = None)

#### NOTE FOUND OUT HAL LIFT OVER CANNOT DEAL WITH . in second to last col, needs a 0 instead ####
# Lift over results located here: /Shares/txpnevol/geba9152 #


3437
14
3423
6


# Create 5p trunc file to run bedtools coverage file

In [6]:
# Make 5p truncated file for coverage calculations
# 1/3 of the gene length and subtract it from the 5p end (max of 750 bp subtraction)

# Dir with the lifted over coordinates
# See: Hal-liftover.sbatch 
transcript_bed_dir = '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs/'

# Outdir
outdir = '/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/annotations/'

def make_5p_trunc_file(transcript_bed_dir, outdir):
    
    # Ensure output directory exists
    os.makedirs(outdir, exist_ok=True)

    # Get all BED files in the directory
    bed_files = glob.glob(os.path.join(transcript_bed_dir, "*.bed"))

    for bed_file in bed_files:
        
        # Extract base name for output file naming (just species)
        base_name = os.path.basename(bed_file).replace(".bed", "").replace("merged.hg38_", "")
            
        # Read in transcript file
        transcript = pd.read_csv(bed_file, sep = "\t", header = None, names = ['chr','start','stop','strand','ROI'])
    
        # Calculate length 
        transcript['length'] = abs(transcript['start'] - transcript['stop'])

        # Calculate 1/3 length 
        transcript['one_third'] = (transcript['length'] / 3).astype(int)

        # Calculate 5p trunc length, rule won't pertain to tiny genes (>300 bps) according to Ru
        # Max of 750
        transcript['trunc_distance_5p'] = transcript.apply(
            lambda row: min(row['one_third'], 750) if row['length'] > 300 else 0,
            axis=1
        )

        ### Calculate start and stops ###
        transcript['start-5p-trunc'] = transcript.apply(
            lambda row: row['start'] + row['trunc_distance_5p'] if row['strand'] == '+' else row['start'],
            axis=1
        )

        transcript['stop-5p-trunc'] = transcript.apply(
            lambda row: row['stop'] - row['trunc_distance_5p'] if row['strand'] == '-' else row['stop'],
            axis=1
        )
        
        # Organize #
        transcript["."] = "."
        transcript = transcript[['chr','start-5p-trunc','stop-5p-trunc','ROI','.','strand']]
        transcript = transcript.sort_values(by=['chr','start-5p-trunc'])
        
        # Save
        transcript.to_csv(outdir + f"5p_trunc_transcripts.{base_name}.halLiftover.refseq.bed", sep = "\t", header = None, index = None)
    
make_5p_trunc_file(transcript_bed_dir, outdir)


# Coverage Filter

In [6]:
# 1. Perform coverage filtering on gene list

# Set outdir
output_dir = "/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/filtered-genes/"
cov_directory = "/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/bedtools-coverage/"

def coverage_filter_genes(metadata_df, cov_directory, output_dir):
    """
    * Takes: 
    - metadata
    - directory with coverage information
    - output dir
    
    * Outputs:
    - filtered gene list    
    """

    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        names = experiment['Names'].split(',')
        coverage_threshold = float(experiment['Sense_Coverage_Cutoff'])
        complexity_cutoff = experiment['Normalized_Complexity_Cutoff']

        
        print(f"Processing experiment: {experiment_name}")
        
        # Initialize variable
        gene_sets = []  # To store genes passing coverage for each sample

        # Process each sample
        for sample in names:
            coverage_file = os.path.join(cov_directory, f"{sample}.3p.combined.txt")
            if not os.path.exists(coverage_file):
                print(f"Coverage file not found for sample {sample}, skipping.")
                continue

            # Read coverage file
            df = pd.read_csv(coverage_file, sep="\t", header=None)
                        
            # Filter genes by coverage threshold
            df_threshold = df[df[9] >= coverage_threshold]
            
            # Add in complexity filter here
            df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
            df_threshold = df_threshold[df_threshold['norm_complexity'] >= complexity_cutoff]
                        
            # Retain only filtered genes
            filtered_genes = set(df_threshold[3])
                        
            print(f"Sample {sample} - Genes after coverage filtering: {len(filtered_genes)}")

            gene_sets.append(filtered_genes)

        # Find common genes across all samples in the experiment
        common_genes = set.intersection(*gene_sets) if gene_sets else set()
        print(f"Sample {experiment_name} - TOTAL genes after ALL coverage filtering: {len(common_genes)}")

        # Save the final list of genes for the experiment
        output_file = os.path.join(output_dir, f"{experiment_name}/{coverage_threshold}_coverage_filtered_genes.txt")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        pd.Series(list(common_genes)).to_csv(output_file, index=False, sep="\t", header=None)

coverage_filter_genes(metadata_df, cov_directory, output_dir)

Processing experiment: BSA-rheMac10
Sample PRO-BSA-Rhesus-1.rheMac10 - Genes after coverage filtering: 3422
Sample PRO-BSA-Rhesus-2.rheMac10 - Genes after coverage filtering: 3559
Sample BSA-rheMac10 - TOTAL genes after ALL coverage filtering: 3415
Processing experiment: BSA-hg38
Sample PRO-BSA-Human-1.hg38 - Genes after coverage filtering: 13323


/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the

Sample PRO-BSA-Human-2.hg38 - Genes after coverage filtering: 13515
Sample BSA-hg38 - TOTAL genes after ALL coverage filtering: 13159
Processing experiment: Meta-BSA-bosTau5
Sample meta_PRO-BSA-Cow.bosTau5 - Genes after coverage filtering: 3603
Sample Meta-BSA-bosTau5 - TOTAL genes after ALL coverage filtering: 3603
Processing experiment: Meta-BSA-nomLeu3
Sample meta_PRO-BSA-Gibbon.nomLeu3 - Genes after coverage filtering: 3743
Sample Meta-BSA-nomLeu3 - TOTAL genes after ALL coverage filtering: 3743
Processing experiment: Meta-BSA-saiBol1
Sample meta_PRO-BSA-SquirrelMonkey.saiBol1 - Genes after coverage filtering: 3822
Sample Meta-BSA-saiBol1 - TOTAL genes after ALL coverage filtering: 3822
Processing experiment: Meta-BSA-rheMac10


/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the

Sample meta_PRO-BSA-Rhesus.rheMac10 - Genes after coverage filtering: 3650
Sample Meta-BSA-rheMac10 - TOTAL genes after ALL coverage filtering: 3650
Processing experiment: Meta-BSA-hg38
Sample meta_PRO-BSA-Human.hg38 - Genes after coverage filtering: 14258
Sample Meta-BSA-hg38 - TOTAL genes after ALL coverage filtering: 14258
Processing experiment: Etchegaray2019histone
Sample SRR9007616.mm10 - Genes after coverage filtering: 1641
Sample Etchegaray2019histone - TOTAL genes after ALL coverage filtering: 1641
Processing experiment: Etchegaray2019histone
Sample SRR9007616.mm10 - Genes after coverage filtering: 1641
Sample Etchegaray2019histone - TOTAL genes after ALL coverage filtering: 1641


/tmp/ipykernel_165888/810746786.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]


# Gene Length Filter

In [7]:
# 2. Length filter before running LIET

# Outdir will be the filtered-genes dir
outdir = '/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/filtered-genes/'

def length_filter_genes(metadata_df, outdir):
    """
    * Takes: 
    - metadata
    - output dir
    
    * Outputs:
    - filtered gene list
    
    """
    
    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Names'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        gene_len_cutoff = float(experiment['Gene_Length_Cutoff'])
        Lifted_Over_Path = experiment['Lifted_Over_Path']
        print(Lifted_Over_Path)
        
        # Load filtered genes for this experiment
        filtered_file_path = os.path.join(outdir, f"{experiment_name}/{coverage_threshold}_coverage_filtered_genes.txt")
#         return filtered_file_path
    
        if not os.path.exists(filtered_file_path):
            print(f"Filtered file not found for {experiment_name}, skipping.")
            continue
        
        # Read the filtered genes
        filtered_genes = pd.read_csv(filtered_file_path, sep="\t", header=None, names=["Gene"])
        starting_len = len(filtered_genes)
        
        print(f"Starting length of gene list: {starting_len}")
        
        # Read in ann
        
        if experiment_name != "BSA-hg38" and experiment_name != "Meta-BSA-hg38":  
            filtered_genes['Gene'] = filtered_genes['Gene'].str.split("|").str[0]
            ann = pd.read_csv(Lifted_Over_Path, sep = "\t", header = None)
            ann.columns = ['chr', 'start','stop','strand','Gene']
            ann['Length'] = abs(ann['start'] - ann['stop'])
            
        else:
            filtered_genes['Gene'] = filtered_genes['Gene'].str.split("|").str[0]
            
            ann = pd.read_csv(Lifted_Over_Path, sep = "\t", header = None)
            ann.columns = ['chr', 'start','stop','Gene','.','strand']
            ann['Length'] = abs(ann['start'] - ann['stop'])
                
        # Merge
#         print(filtered_genes)
        merge = ann.merge(filtered_genes, on = 'Gene' )
        merge_len = len(merge)
                        
#         print(f"Merged length of gene list: {merge_len}")
        
        # Filter based on thresh
        df_filtered = merge[merge["Length"] >= gene_len_cutoff]
                        
        df_filtered_len = len(df_filtered)
    
        print(f"Sample {experiment_name} - TOTAL genes after coverage/length filtering: {df_filtered_len}")

        df_filtered = df_filtered['Gene']
        
        # Save
        output_file = os.path.join(outdir, f"{experiment_name}/{coverage_threshold}_coverage_length_filtered_genes.txt")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        df_filtered.to_csv(output_file, index=False, sep="\t", header=None)

length_filter_genes(metadata_df, outdir)

/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_rheMac10.bed
Starting length of gene list: 3415
Sample BSA-rheMac10 - TOTAL genes after coverage/length filtering: 2975
/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/ortholog-output/MANE-annotation-overlaps-across-all-species/overlapping_orthologs_Hal_format.MANE.bed
Starting length of gene list: 13159
Sample BSA-hg38 - TOTAL genes after coverage/length filtering: 2281
/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_bosTau5.bed
Starting length of gene list: 3603
Sample Meta-BSA-bosTau5 - TOTAL genes after coverage/length filtering: 3079
/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_nomLeu3.bed
Starting length of gene list: 3743
Sample Meta-BSA-nomLeu3 - TOTAL genes after coverage/length filtering: 3277
/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordin

In [31]:
list(metadata_df['Lifted_Over_Path'])

['/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_rheMac10.bed',
 '/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/ortholog-output/MANE-annotation-overlaps-across-all-species/overlapping_orthologs_Hal_format.MANE.bed',
 '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_bosTau5.bed',
 '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_nomLeu3.bed',
 '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_saiBol1.bed',
 '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-orthologs//merged.hg38_rheMac10.bed',
 '/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/ortholog-output/MANE-annotation-overlaps-across-all-species/overlapping_orthologs_Hal_format.MANE.bed',
 '/Users/geba9152/3end_paper_backed_up_data/MANE-lifted-over-coordinates-BUSCO-ort

# Make LIET Annotation Files

In [10]:
# metadata_df = metadata_df.loc[metadata_df['Experiment'] == "Meta-BSA-hg38"]
# metadata_df

In [8]:
def make_3_p_liet_ann_and_pad(metadata_df, outdir):
    '''
    * Takes: 
    - metadata
    - output dir
    
    * Outputs:
    - filtered gene list-->3p liet.ann format (LIET input) / pad files
    '''
    
    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Samples'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        Lifted_Over_Path = experiment['Lifted_Over_Path']
        
        if experiment_name != "BSA-hg38" and experiment_name != "Meta-BSA-hg38": 
            transcript_ann = pd.read_csv(Lifted_Over_Path, sep="\t", header=None, names=['chr','start','stop','strand','gene'])
        else:
            transcript_ann = pd.read_csv(Lifted_Over_Path, sep="\t", header=None, names=['chr','start','stop','gene','.','strand'])
            transcript_ann['gene'] = transcript_ann['gene'].str.split("|").str[0]
        
        # Remove genes with missing chromosome values
        missing_chr = transcript_ann[transcript_ann['chr'].isna()]
        if not missing_chr.empty:
            print(f"Removing {len(missing_chr)} genes due to missing chromosome values.")
        transcript_ann = transcript_ann.dropna(subset=['chr'])
        
        filtereddir = '/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/filtered-genes/'
        filtered_file_path = os.path.join(filtereddir, f"{experiment_name}/{coverage_threshold}_coverage_length_filtered_genes.txt")
        filtered_genes = pd.read_csv(filtered_file_path, sep="\t", header=None, names=["gene"])
        
        if experiment_name == "BSA-hg38":
            filtered_genes['gene'] = filtered_genes['gene'].str.split("|").str[0]
            
        elif experiment_name == "Meta-BSA-hg38":
            filtered_genes['gene'] = filtered_genes['gene'].str.split("|").str[0]
        
        filtered_genes_ann = filtered_genes.merge(transcript_ann, on="gene")
        filtered_genes_ann['length'] = abs(filtered_genes_ann['stop'] - filtered_genes_ann['start'])
                
        duplicates = filtered_genes_ann[filtered_genes_ann['gene'].duplicated(keep=False)]
        if not duplicates.empty:
            print("Duplicate genes found:")
            print(duplicates)
        
        filtered_genes_ann = filtered_genes_ann[~filtered_genes_ann['gene'].isin(duplicates)]
        filtered_genes_liet_ann = filtered_genes_ann[['chr','start','stop','gene','length','strand']].sort_values(by=['chr', 'start'])
        
        print(f"Sample {experiment_name} - Genes in liet.ann: {len(filtered_genes_liet_ann)}")
        
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-3p-UTR.liet.ann")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_liet_ann.to_csv(output_file, index=False, sep="\t", header=None)
        
        filtered_genes_ann['.'] = "."
        filtered_genes_bed = filtered_genes_ann[['chr','start','stop','gene','.','strand']].sort_values(by=['chr', 'start'])
        
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-3p-UTR.bed")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_bed.to_csv(output_file, index=False, sep="\t", header=None)
        
        filtered_genes_ann = filtered_genes_ann.sort_values(by=['chr', 'start'])
        filtered_genes_ann['pad'] = '3000,20000'
        filtered_genes_pad = filtered_genes_ann[['gene','pad']]
        
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-3p-UTR.pad")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_pad.to_csv(output_file, index=False, sep="\t", header=None)

outdir = "/scratch/Users/geba9152/LIET-3end-analysis/daniel-animal-liet/annotations/"
make_3_p_liet_ann_and_pad(metadata_df, outdir)


Sample BSA-rheMac10 - Genes in liet.ann: 2975
Duplicate genes found:
          gene    chr      start       stop  . strand  length
536       KRAS  chr12   25205246   25250929  0      -   45683
537       KRAS  chr12   25205246   25250929  0      -   45683
1215    INPP4A   chr2   98444858   98594392  0      +  149534
1216    INPP4A   chr2   98444858   98594392  0      +  149534
1276     ITGA6   chr2  172427586  172506459  0      +   78873
1277     ITGA6   chr2  172427586  172506459  0      +   78873
1733     MOCS2   chr5   53095679   53109757  0      -   14078
1734     MOCS2   chr5   53095679   53109757  0      -   14078
1797     P4HA2   chr5  132190147  132227853  0      -   37706
1798     P4HA2   chr5  132190147  132227853  0      -   37706
2097  SLC39A14   chr8   22367278   22422736  0      +   55458
2098  SLC39A14   chr8   22367278   22422736  0      +   55458
Sample BSA-hg38 - Genes in liet.ann: 2281
Removing 127 genes due to missing chromosome values.
Sample Meta-BSA-bosTau5 - Gene